# LangChain — Intermediate Agent

**Framework:** LangChain (LCEL)  
**Level:** 02 — Intermediate  
**Model:** `gemini-2.0-flash`

### What's New vs Simple
| Feature | Simple | Intermediate |
|---|---|---|
| Tools | 2 | **3** (added `get_travel_advisory`) |
| Memory | None | **`RunnableWithMessageHistory`** |
| Output | Free text | **Structured** (Pydantic) |
| Session | Stateless | **Session-aware** via `session_id` |

## Concept: RunnableWithMessageHistory

```
LangChain Memory Architecture:

┌─────────────────────────────────────────────────────┐
│            RunnableWithMessageHistory               │
│                                                     │
│  invoke(input, config={session_id: "abc"})          │
│       │                                             │
│       ▼                                             │
│  get_session_history("abc")                         │
│       │ returns ChatMessageHistory                  │
│       │                                             │
│       ▼                                             │
│  Injects history into prompt's {chat_history}       │
│       │                                             │
│       ▼                                             │
│  AgentExecutor runs with full context               │
│       │                                             │
│       ▼                                             │
│  Appends new messages back to session history       │
└─────────────────────────────────────────────────────┘

Different session_id = isolated conversation thread
```

**Key components:**
- `ChatMessageHistory` — in-memory per-session message store
- `get_session_history(id)` — callable that returns the history for a session
- `RunnableWithMessageHistory` — wraps any runnable, auto-reads/writes history
- `MessagesPlaceholder("chat_history")` — the slot in the prompt where history is injected

## Setup

In [ ]:
import os
from pydantic import BaseModel, Field
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.runnables.history import RunnableWithMessageHistory

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in your .env"
print("✓ Ready")

## Structured Output Schema

In [ ]:
class TravelBriefing(BaseModel):
    city: str = Field(description="Name of the city")
    weather_summary: str = Field(description="Weather conditions and temperature")
    local_time: str = Field(description="Current local time with timezone")
    travel_advisory: str = Field(description="Safety advisory level and notes")
    recommendation: str = Field(description="One-sentence travel recommendation")

print("Schema:", list(TravelBriefing.model_fields.keys()))

## 3 Tools

In [ ]:
@tool
def get_weather(city: str) -> dict:
    """Return weather conditions for a city. Use when asked about weather or temperature."""
    data = {
        "london":    {"condition": "Cloudy",       "temp_c": 12, "humidity": 78},
        "new york":  {"condition": "Sunny",         "temp_c": 22, "humidity": 45},
        "bangalore": {"condition": "Rainy",         "temp_c": 25, "humidity": 85},
        "tokyo":     {"condition": "Clear",         "temp_c": 18, "humidity": 60},
        "paris":     {"condition": "Partly Cloudy", "temp_c": 16, "humidity": 65},
    }
    key = city.lower()
    if key in data:
        d = data[key]
        return {"city": city, "condition": d["condition"],
                "temperature_celsius": d["temp_c"], "humidity_percent": d["humidity"]}
    return {"error": f"No data for '{city}'."}


@tool
def get_time(city: str) -> dict:
    """Return current local time for a city."""
    times = {
        "london": "14:30 GMT", "new york": "09:30 EST",
        "bangalore": "20:00 IST", "tokyo": "22:30 JST", "paris": "15:30 CET",
    }
    key = city.lower()
    if key in times:
        return {"city": city, "local_time": times[key]}
    return {"error": f"No time data for '{city}'."}


@tool
def get_travel_advisory(city: str) -> dict:  # ← NEW
    """Return travel safety advisory for a city. Use when asked about safety or advisories."""
    advisories = {
        "london":    {"level": "Low",    "notes": "Routine precautions."},
        "new york":  {"level": "Low",    "notes": "Normal precautions."},
        "bangalore": {"level": "Medium", "notes": "Monsoon affects transport."},
        "tokyo":     {"level": "Low",    "notes": "Very safe city."},
        "paris":     {"level": "Low",    "notes": "Alert in crowded spots."},
    }
    key = city.lower()
    if key in advisories:
        a = advisories[key]
        return {"city": city, "advisory_level": a["level"], "notes": a["notes"]}
    return {"error": f"No advisory data for '{city}'."}


tools = [get_weather, get_time, get_travel_advisory]
print(f"{len(tools)} tools defined")

## Agent + Memory Wiring

Three steps:
1. Add `MessagesPlaceholder("chat_history")` to the prompt
2. Build the executor as usual
3. Wrap with `RunnableWithMessageHistory` — pass the session history getter

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a professional travel assistant. When asked about a city, always check "
     "weather, local time, AND travel advisory. Remember the conversation history."),
    MessagesPlaceholder("chat_history"),   # ← history injected here
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=False)

# Session store: maps session_id → ChatMessageHistory
_store: dict[str, BaseChatMessageHistory] = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in _store:
        _store[session_id] = ChatMessageHistory()
    return _store[session_id]

# Wrap: auto-reads history before invoke, auto-writes after
agent_with_memory = RunnableWithMessageHistory(
    executor,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

print("Agent with memory ready")

## Multi-Turn Run

In [ ]:
def run(query: str, session_id: str = "session_01") -> str:
    result = agent_with_memory.invoke(
        {"input": query},
        config={"configurable": {"session_id": session_id}},
    )
    return result["output"]


SESSION = "travel_session"

# Turn 1
q1 = "Give me a full travel briefing for Tokyo."
print(f"User: {q1}")
print(f"Agent: {run(q1, SESSION)}\n")

In [ ]:
# Turn 2 — agent now has Tokyo context
q2 = "Now do the same for Bangalore. I prefer warm weather — which would you recommend?"
print(f"User: {q2}")
print(f"Agent: {run(q2, SESSION)}\n")

In [ ]:
# Turn 3 — tests cross-turn recall
q3 = "Based on what you told me, what's the safer city to visit?"
print(f"User: {q3}")
print(f"Agent: {run(q3, SESSION)}\n")

In [ ]:
# Inspect memory contents
history = get_session_history(SESSION)
print(f"Messages in memory: {len(history.messages)}")
for i, msg in enumerate(history.messages):
    print(f"  [{i}] {type(msg).__name__}: {str(msg.content)[:80]}")

In [ ]:
# Different session = fresh conversation
q_new = "What did you tell me about Tokyo?"
print(f"New session — User: {q_new}")
print(f"Agent: {run(q_new, 'fresh_session')}")
# Agent should say it has no prior context

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| `MessagesPlaceholder('chat_history')` | Reserved slot — LangChain injects history here automatically |
| `get_session_history(id)` | Factory function — one `ChatMessageHistory` per session |
| `RunnableWithMessageHistory` | Wraps any runnable; handles read/write of history transparently |
| `config={"session_id": ...}` | Thread/session routing — same ID = shared memory |
| Fresh session = no memory | Sessions are isolated — good for multi-user apps |

### Comparison Across Frameworks (Memory)
| Framework | Memory Mechanism |
|---|---|
| **ADK** | `InMemorySessionService` + reuse `session_id` |
| **LangChain** | `RunnableWithMessageHistory` + `ChatMessageHistory` |
| **LangGraph** | `MemorySaver` checkpointer + `thread_id` in config |
| **CrewAI** | Task `context=[prev_task]` (explicit dependency, not conversation history) |

### Next: [03-Complex →](../03-complex/agent.ipynb)